In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import sys, os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import defaultdict
from scipy.stats import gaussian_kde
sys.path.append(os.path.abspath(".."))
from models.utils_common import population as pop
import utils

### Data analysis

#### Temperature Zones

In [ ]:
### Map of the temperature zones

wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
era5_tz = xr.open_dataset(f"{wdir}/data/temperature_zones/ERA5_mean_1980-2019_land_t2m_tz.nc").t2m
era5_tz.plot()

#### TMRELs

##### Population weighted PDFs of TMRELs vs Temperature Zones

In [ ]:
### Open all files and calculate KDE

wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
years = [1990,2010,2020]

# Import population maps for years with TMRELs
population = pop.LoadPopulationMap(wdir, scenario="SSP2_ERA5", ssp="SSP2", years=[1990,2020])
population = population.sel(time=["1990", "2010", "2020"]).mean("time")

# Import TMRELs maps
tmrel = {}
for year in years:
    tmrel[year] = xr.open_dataset(wdir+f"/data/TMRELs_nc/TMRELs_{year}.nc")

# Import temperature zones
era5_tz = xr.open_dataset(wdir+f"/data/temperature_zones/ERA5_mean_1980-2019_land_t2m_tz.nc")

# Initialize dicitonaries for counts and weights for KDE
counts_tmrel = defaultdict(dict); weights_tmrel = defaultdict(dict)

for year in years:
    for tz in range(6,29):
        counts_tmrel[year][tz], weights_tmrel[year][tz] = utils.WeightsAndCountsForKDE(tmrel[year], era5_tz, population,tz)

In [ ]:
### Plot TMRELs vs TZ for one year
year = 2010
tz_values = list(range(6, 29))
colors = cm.copper(np.linspace(0, 1, len(tz_values)))

fig, ax = plt.subplots(figsize=(10, 8))#, dpi=300)

for i, tz in enumerate(tz_values):
    data = counts_tmrel[year][tz]
    w = weights_tmrel[year][tz]

    if len(data) > 50000:
        idx = np.random.choice(len(data), size=50000, replace=False, p=w)
        data = data[idx]
        w = w[idx]
        w = w / w.sum()

    kde = gaussian_kde(data, weights=w)

    p5, p95 = np.percentile(data, [5, 95])
    x = np.linspace(p5, p95, 200)
    y = kde(x)

    scale = 0.4 / y.max()
    y_scaled = y * scale

    ax.fill_betweenx(x, tz, tz + y_scaled, color=colors[i], alpha=0.8, zorder=2)

utils.StylizeAxes(ax, xticks=tz_values, xtickslabels=tz_values, xlim=(5.5,28.5), ylim=(14,30), xlabel='Temperature zone [°C]',
             ylabel="TMREL [°C]", title=f"Population-weighted TMREL distributions by Temperature Zone for {year}\n5–95 Percentile Range\n",
             grid=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

In [ ]:
### Plot TMRELs vs TZ for all years

tz_values = list(range(6, 29))
colors = cm.copper(np.linspace(0, 1, len(tz_values)))

fig, axs = plt.subplots(1,3, figsize=(18, 5))#, dpi=300)
axs=axs.flatten()

for j,year in enumerate([1990,2010,2020]):
    for i, tz in enumerate(tz_values):
        data = counts_tmrel[year][tz]
        w = weights_tmrel[year][tz]

        if len(data) > 50000:
            idx = np.random.choice(len(data), size=50000, replace=False, p=w)
            data = data[idx]
            w = w[idx]
            w = w / w.sum()

        kde = gaussian_kde(data, weights=w)

        p5, p95 = np.percentile(data, [5, 95])
        x = np.linspace(p5, p95, 200)
        y = kde(x)

        scale = 0.4 / y.max()
        y_scaled = y * scale

        axs[j].fill_betweenx(x, tz, tz + y_scaled, color=colors[i], alpha=0.9, zorder=2)

    utils.StylizeAxes(axs[j], xticks=tz_values, xtickslabels=tz_values, xlim=(5.5,28.5), ylim=(13.5,30), xlabel='Temperature zone [°C]',
                ylabel="TMREL [°C]", title=f"{year}", grid=True, grid_kwargs={'axis':'both', 'color':'white', 'zorder':0}, facecolor='whitesmoke')
plt.suptitle('Population-weighted TMREL Distributions by Temperature Zone')